# Random Per-Cell 14 px-Patch Stimulation

Single-cell optogenetic stimulation experiment using `RandomStimPerCell14pxPatches`.
For every segmented cell each FOV is tiled into a regular grid of 14 px patches; only
patches that are entirely inside the cell mask are kept, and one is chosen at random
to receive a 10 px-diameter dot stimulation. The selected patch (and its bounding
box) is written to `stim_patches/{fname}.parquet` so post-processing can re-extract
image features for that exact patch and correlate them with the local cell response.

**Workflow overview:**
1. Initialise the DMD-equipped microscope and define imaging / stim / optocheck channels.
2. Configure the pipeline with Cellpose segmentation, the new per-cell patch stimulator, ERK-KTR feature extraction, and tracking.
3. Pick FOV positions in napari and build the acquisition as a single RTMSequence phase: 5 min baseline, single stim pulse, 15 min response, optocheck on the last frame.
4. Validate and run the experiment (~24 FOVs in parallel, 30 s intervals, 20 min total).
5. Post-process: merge per-FOV tracks into `exp_data.parquet` and merge the patch sidecar files into `stim_patches.parquet`.


In [ ]:
import os
import time
from faro.core.data_structures import (
    Channel,
    PowerChannel,
    RTMSequence,
    SegmentationMethod,
    combine,
)
import faro.core.utils as utils

### Experimental Settings

**Microscope:** Moench (DMD-based). Per-cell single-cell patterning needs the DMD.

**What to customise below:**
- `SLEEP_BEFORE_EXPERIMENT_START_in_H` -- delay before acquisition starts (hours).
- `base_path` / `experiment_name` -- output directory. All data (zarr, tracks, TIFFs, sidecar parquets) is saved under this path.
- `INTERVAL_S`, `BASELINE_MIN`, `RESPONSE_MIN` -- experiment timing.
- `stim_channel` -- the stimulation light channel. Adjust `power` (0-100) and `exposure` (ms).
- `imaging_channels` -- channels acquired every timepoint. Channel 0 is used for segmentation.
- `optocheck_channel` -- reference channel acquired on the last frame to verify optogenetic tool expression.


In [ ]:
from faro.microscope.pertzlab.moench import Moench

mic = Moench(None)
mic.mmc.setChannelGroup("TTL_ERK")  # adjust to the channel group used in MicroManager

In [ ]:
## Configuration
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0  # delay before acquisition (hours); 0 = start immediately

INTERVAL_S = 30          # 30 s between frames

# Frame counts per phase (all 0-indexed inside a single phase)
N_BASELINE = 20          # imaging frames before the stim block
N_STIM     = 10          # consecutive stim frames (mask stays fixed across these)
N_RECOVERY = 60          # imaging frames after the stim block
N_FRAMES   = N_BASELINE + N_STIM + N_RECOVERY

# Stim frame indices: a contiguous block right after baseline.
STIM_FRAMES = list(range(N_BASELINE, N_BASELINE + N_STIM))

## Storage -- all output (zarr, tracks, TIFFs, stim_patches sidecar) goes under this directory
base_path = r"Z:\lhinder\data\rtm_mm_data\exp_427"
experiment_name = "random_stim_per_cell_14px_patches"
path = os.path.join(base_path, experiment_name)

## Stimulation channel -- light used for optogenetic activation
stim_channel = PowerChannel(
    config="CyanStim",
    exposure=300,
    group="TTL_ERK",
    power=20,
)

## Imaging channels -- order matters (channel 0 is used for segmentation)
imaging_channels = (
    PowerChannel(config="mScarlet3", exposure=100, group="TTL_ERK", power=5),   # ERK-KTR reporter
    PowerChannel(config="miRFP", exposure=1000, group="TTL_ERK", power=99),       # PIP

)

## Optocheck channel -- acquired only on ref_frames; longer exposure to measure optogenetic tool expression
optocheck_channel = PowerChannel(config="mCitrine", exposure=300, group="TTL_ERK", power=30)

print(f"Experiment timing: {N_FRAMES} frames @ {INTERVAL_S}s = {(N_FRAMES-1)*INTERVAL_S/60:.1f} min")
print(f"Baseline : frames 0..{N_BASELINE-1} ({N_BASELINE*INTERVAL_S/60:.1f} min)")
print(f"Stim     : frames {STIM_FRAMES[0]}..{STIM_FRAMES[-1]} ({N_STIM*INTERVAL_S/60:.1f} min)")
print(f"Recovery : frames {STIM_FRAMES[-1]+1}..{N_FRAMES-1} ({N_RECOVERY*INTERVAL_S/60:.1f} min)")
print(f"Optocheck: last frame ({N_FRAMES-1})")


### Pipeline Setup

`RandomStimPerCell14pxPatches` divides each FOV into 14 px patches and stamps a 10 px-diameter
dot onto one randomly selected patch per cell. Selection is seeded by `(fov, fov_timestep)` so
re-running post-processing reproduces the exact patches that were illuminated live. Patch
coordinates are accumulated in-memory while the experiment runs and merged directly into
`exp_data.parquet` after acquisition (no separate sidecar file).

**Components below:**
- **Segmentation:** Cellpose v4 on imaging channel 0 (nuclear marker). `save_tracked=True` writes tracked label masks alongside raw segmentation.
- **Stimulator:** `RandomStimPerCell14pxPatches(seed=0)`.
- **Feature extraction:** `FE_ErkKtr("labels")` for cytoplasmic/nuclear ERK-KTR ratio.
- **Tracker:** `TrackerTrackpy(search_range=50)` (max 50 px displacement between frames).
- **Optocheck:** `OptoCheckFE(used_mask="labels")` measures optogenetic reporter intensity on `ref_frames`.


In [ ]:
from faro.stimulation.random_stim_per_cell_14px_patches import RandomStimPerCell14pxPatches
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=CellposeV4(),
        use_channel=0,
        save_tracked=True,
    )
]

stimulator = RandomStimPerCell14pxPatches(seed=0)
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=50)
optocheck = OptoCheckFE(used_mask="labels")

from faro.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,
)

from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)

### GUI

Open the napari viewer with the MicroManager widget. Use the MDA widget to pick the ~24 FOV positions for `generate_fov_positions` below.


In [ ]:
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)

### DMD calibration

The DMD must be calibrated against the camera before running a single-cell stim experiment so that the patch dots land on the right pixels.


In [ ]:
mic.dmd.affine = None
try:
    mm_wdg._core_link.cleanup()
except Exception as e:
    print(e)
import pymmcore_plus

pymmcore_plus.configure_logging(stderr_level="CRITICAL")
mic.calibrate_dmd(verbose=True,radius=4)
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

# Use this to focus the DMD (will activate your sample!)

In [ ]:
# --- Cell 6a: arm DMD checkerboard for live focus ---
# Disable OverlapMode so the DMD latches the pattern continuously
# instead of waiting for per-frame camera TTL. OverlapMode=On is
# correct during the real MDA (tests pass under it), but napari's
# Live view doesn't reliably produce the TTL pulses the DMD expects,
# so in that context ``displaySLMImage`` behaves as a one-shot flash
# and the DMD sits in an unpredictable idle state between camera
# frames. Flipping to OverlapMode=Off makes the pattern visible
# throughout Live.
_prev_overlap = mic.mmc.getProperty(mic.dmd.name, "OverlapMode")
mic.mmc.setProperty(mic.dmd.name, "OverlapMode", "Off")

#£mic.mmc.setConfig(CHANNEL_GROUP, STIM_CHANNEL)
#mic.mmc.setExposure(STIM_EXPOSURE_MS)
# Long SLM exposure so the pattern stays on as napari Live streams.
mic.mmc.setSLMExposure(mic.dmd.name, 200000)
try:
    mic.mmc.setProperty("LED", "Cyan_Level", 1)
except Exception as e:
    print(f"(skipping LED power set: {e})")
mic.dmd.checker_board(pixels=100)
print(f"DMD checker armed (OverlapMode was {_prev_overlap!r}, now Off).")
print("Press Live in napari-mm and refocus. Run the next cell to disarm.")

In [ ]:
# --- Cell 6b: disarm DMD + restore OverlapMode ---
mic.dmd.all_off()
try:
    mic.mmc.setProperty(mic.dmd.name, "OverlapMode", _prev_overlap)
    print(f"DMD blanked, OverlapMode restored to {_prev_overlap!r}.")
except NameError:
    print("DMD blanked. (OverlapMode state unknown — arm cell wasn't run.)")

### Build acquisition events

Single-phase experiment: 41 frames at 30 s intervals (20 min total). Stim fires once at frame 10
(after the 5 min baseline) and the optocheck channel is acquired on the last frame.

`apply_fov_batching(events, time_per_fov=2.0)` keeps the timing valid even if 24 FOVs cannot all be
imaged within one 30 s interval -- FOVs are split into sequential sub-batches automatically.


In [ ]:
fov_positions = utils.generate_fov_positions(mic, viewer=viewer)

phase = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=fov_positions,
    channels=imaging_channels,
    stim_channels=(stim_channel,),
    stim_frames=STIM_FRAMES,
    ref_channels=(optocheck_channel,),
    ref_frames=[-1],
    rtm_metadata={
        "phase_name": "RandomPatchStim",
        "phase_id": 0,
        "treatment_name": "Random per-cell 14px-patch stimulation",
    },
)

# combine() flattens the RTMSequence into per-event records that the controller and
# apply_fov_batching can consume; passing axis="t" is a no-op for a single phase.
events = combine(phase, axis="t")
events = utils.apply_fov_batching(events, time_per_fov=5.0)

print(f"Total events: {len(events)}")
print(f"FOVs: {len(fov_positions)}")
utils.events_to_dataframe(events).sort_values("timestep")


In [ ]:
events

### Validate Experiment

`validate_events` checks FOV positions, channel configs, and pipeline compatibility (in particular,
that the new stim class accepts the right call signature and that no required metadata is missing).


In [ ]:
ctrl = Controller(mic, pipeline, writer=writer)
ctrl.validate_events(events)

### Run experiment

Optional sleep loop delays acquisition start (`SLEEP_BEFORE_EXPERIMENT_START_in_H`).
Disconnect the napari live view before the MDA engine takes over the camera.


In [ ]:
for _ in range(0, int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()
except Exception:
    pass

ctrl.run_experiment(events, stim_mode="current")

### Post-processing

After acquisition completes:

1. `finish_experiment()` flushes the pipeline queue and closes the zarr writer.
2. `generate_exp_data_from_tracks()` merges per-FOV tracks into a single `exp_data.parquet`.
3. The patch coordinates accumulated by the stimulator are merged directly into `exp_data.parquet` on `(fov, fov_timestep, label)`. Stim-frame rows for stimulated cells gain `patch_y_min`, `patch_x_min`, `patch_y_max`, `patch_x_max`, `patch_dot_y`, `patch_dot_x`, etc.; all other rows get NaN for these columns.
4. Reconnect the napari live view.


In [ ]:
import pandas as pd

ctrl.finish_experiment()
utils.generate_exp_data_from_tracks(path)

# Merge the patch coordinates accumulated by the stimulator into exp_data.parquet,
# joining on (fov, fov_timestep, label). Drop any duplicate stim calls per cell.
exp_data_path = os.path.join(path, "exp_data.parquet")
df_exp = pd.read_parquet(exp_data_path)
df_patches = stimulator.to_dataframe().drop_duplicates(
    subset=["fov", "fov_timestep", "label"]
)
if not df_patches.empty:
    df_exp = df_exp.merge(
        df_patches, on=["fov", "fov_timestep", "label"], how="left"
    )
df_exp.to_parquet(exp_data_path)
print(f"Merged {len(df_patches)} patch rows into {exp_data_path}")

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
df_exp